# Feature Engineering

## Import

In [ ]:
import pyspark
from pyspark.ml import Pipeline, Estimator, Model
from pyspark.ml.util import DefaultParamsReadable, DefaultParamsWritable
from pyspark.ml.feature import VectorAssembler
#from synapse.ml.isolationforest import IsolationForest
from pyspark.sql.functions import first, sum, col, max, min
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
import os
os.environ['HADOOP_HOME'] = "C:\\hadoop" 
# Добавляем путь к bin в системный PATH
os.environ['PATH'] += os.pathsep + "C:\\hadoop\\bin"

In [2]:
def finalize_parquet(folder_path, target_name):
    files = os.listdir(folder_path)
    # Добавляем [0] для получения строки
    part_file = [f for f in files if f.startswith("part-") and f.endswith(".parquet")][0]
    
    # Определяем путь к папке joined (на уровень выше временной папки)
    parent_dir = os.path.dirname(folder_path) 
    # Формируем полный путь: "../datasets/joined/train_data.parquet"
    final_path = os.path.join(parent_dir, target_name)
    
    # Если такой файл уже есть, os.rename может выдать ошибку, лучше удалить старый
    if os.path.exists(final_path):
        os.remove(final_path)

    os.rename(os.path.join(folder_path, part_file), final_path)
    
    # Очистка
    for f in os.listdir(folder_path):
        os.remove(os.path.join(folder_path, f))
    os.rmdir(folder_path)
    

In [3]:
train_files = ['../datasets/train/train_part_1.parquet','../datasets/train/train_part_2.parquet','../datasets/train/train_part_3.parquet']
train_path = '../datasets/train/'
# создание spark-сессии
spark =SparkSession.builder \
    .appName("feature_eng") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "1g") \
    .getOrCreate()
# загрузка исходных данных и списка релевантных для задачи признаков в объект DataFrame
df= spark.read.parquet(*train_files)

    # .config("spark.jars.packages", "com.microsoft.azure:synapseml_2.12:1.1.2") \
    # .config("spark.jars.repositories", "https://mmlspark.azureedge.net/maven") \


In [ ]:
df.columns

In [ ]:
max_date = df.select(F.max("event_dttm")).collect()[0][0]
print(f"Max Date {max_date}")

In [ ]:
min_date = df.select(F.min("event_dttm")).collect()[0][0]
print(f"Max Date {min_date}")

## Джойн лейблов + заполнение пропусков

In [ ]:
df_labels = spark.read.parquet(r'../datasets/train/train_labels.parquet')
df = df.join(other=df_labels,on=['customer_id', 'event_id'], how='left')

df = df.na.fill({
    "target": 0,
    # "label_category": "unknown"
})

## ЗАГРУЗКА TEST PARQUET

In [ ]:
test_path = "../datasets/test/test.parquet"
pretest_path = "../datasets/test/pretest.parquet"
df_test = spark.read.parquet(test_path)
df_pretest = spark.read.parquet(pretest_path)

## ЗАГРУЗКА PRETRAIN

In [ ]:
pre_train_paths = ["../datasets/pretrain/pretrain_part_1.parquet", "../datasets/pretrain/pretrain_part_2.parquet", "../datasets/pretrain/pretrain_part_3.parquet"]
df_pretrain = spark.read.parquet(*pre_train_paths)

## PIPELINE БОГА 

In [ ]:
# 1. КЛАСС-МОДЕЛЬ: Просто приклеивает готовую таблицу агрегатов
class UserHistoryModel(Model, DefaultParamsReadable, DefaultParamsWritable):
    def __init__(self, user_profile):
        super(UserHistoryModel, self).__init__()
        self.user_profile = user_profile

    def _transform(self, dataset):
        # Джойним профили из истории к текущим событиям
        enriched_df = dataset.join(self.user_profile, on="customer_id", how="left")
        
        # Заполняем нулями тех, кого не было в истории
        return enriched_df.fillna(0, subset=["hist_event_cnt", "hist_amt_sum"])

# 2. КЛАСС-ОБУЧАТЕЛЬ: Вычисляет агрегаты по переданной истории
class UserHistoryAggregator(Estimator, DefaultParamsReadable, DefaultParamsWritable):
    def __init__(self, history_df):
        super(UserHistoryAggregator, self).__init__()
        self.history_df = history_df

    def _fit(self, dataset):
        # Здесь происходит тяжелая работа: расчет профиля по истории
        user_profile = self.history_df.groupBy("customer_id").agg(
            F.count("event_id").alias("hist_event_cnt"),
            F.sum("operaton_amt").alias("hist_amt_sum"),
            F.max("event_dttm").alias("hist_last_dt")
        )
        
        # Кэшируем результат, чтобы transform работал мгновенно
        user_profile.cache()
        
        return UserHistoryModel(user_profile=user_profile)

## РАЗДЕЛЕНИЕ НА ТРЕНИРОВОЧНУЮ И ВАЛИДАЦИОННУЮ 

In [ ]:
fractions = {row[0]: 0.8 for row in df.select("target").distinct().collect()}

df_train = df.stat.sampleBy("target", fractions, seed=42)
df_valid = df.join(df_train, on="event_id", how="left_anti")


## ВКЛ И ВЫКЛ ПАЙПЛАЙНА

In [ ]:
# 1. Готовим "Базы знаний" (Историю)
# Для трейна и валидации — только претрейн
history_train = df_pretrain
history_full = df_pretrain.unionByName(df_pretest).distinct()

# --- ШАГ 1: ТРЕНИРОВКА (Валидация) ---
# Создаем агрегатор, который видит только претрейн
agg_train = UserHistoryAggregator(history_df=history_train)
pipeline_train = Pipeline(stages=[agg_train])

# Обучаем и применяем (Transform)
model_train = pipeline_train.fit(df_train)
df_train = model_train.transform(df_train)
df_valid = model_train.transform(df_valid)

# --- ШАГ 2: ТЕСТ (Продакшен-режим) ---
# Создаем ТАКОЙ ЖЕ агрегатор, но с ОБНОВЛЕННОЙ историей
agg_test = UserHistoryAggregator(history_df=history_full)
pipeline_test = Pipeline(stages=[agg_test])

# Обучаем заново (fit), чтобы зафиксировать новую таблицу агрегатов
model_test = pipeline_test.fit(df_test)
df_test = model_test.transform(df_test)

## ВЫГРУЗКА ФАЙЛОВ АЙОУ

In [ ]:
# 2. Сохранение во временные папки с защитой от OOM (repartition)
train_temp = "../datasets/joined/temp_train_folder"
valid_temp = "../datasets/joined/temp_valid_folder"
test_temp = "../datasets/joined/temp_test_folder"

df_train.repartition(1).write.mode("overwrite").parquet(train_temp)
df_valid.repartition(1).write.mode("overwrite").parquet(valid_temp)
df_test.repartition(1).write.mode("overwrite").parquet(test_temp)

In [ ]:
finalize_parquet(train_temp, "train_data.parquet")
finalize_parquet(valid_temp, "valid_data.parquet")
finalize_parquet(test_temp, "test_data.parquet")

print("Готово! Созданы файлы train_data.parquet, valid_data.parquet, test_data.parquet ")

## ВЫХОДНОЙ СПИСОК КОЛОНОК 

In [ ]:
with open('../datasets/joined/columns_list.txt',mode='w') as file_ik:
    for i in range(len(df_train.columns)):
        if i != len(df_train.columns)-1:
            end = ','
        else:
            end = ''
        file_ik.write(df_train.columns[i]+end)
